# 37 · Regression with several strata columns, checked against statsmodels

## What this checks

**Regression** fits a model of weekly dengue cases. This check covers release 0.9.0, and one feature of it: strata
made from several columns together.

A stratum is a group of rows compared only with each other. The Brazil dengue models compare weeks only within the
same municipality, year and month. So Regression takes the three columns together, and each combination of them is
one stratum. Comparing within strata keeps season and place from looking like an effect of the weather.

If the strata were built wrongly, every estimate would compare the wrong weeks.

## How it is checked

We fit the same model with **statsmodels**, a Python library for statistical models. It shares no code with
Fieldwork, and it reaches the answer another way.

- **Fieldwork** absorbs the strata in a conditional Poisson model. A Poisson model is a regression for counts.
  "Conditional" means each stratum's own level is set aside, so only the comparison within it is estimated.
- **statsmodels** instead adds one yes-or-no column per stratum. This gives the same slopes by a different road.
- **pandas** builds the strata here from the three columns. It does not use Fieldwork's own key.

The data are two municipalities of Rio de Janeiro state, 2010 to 2025. Twelve weeks that appear twice at the turn of
a year are left out. The model is weekly cases on the dataset's weekly air temperature, `tempmed`, adjusted
for weekly rainfall, `precip_total_semana`. The errors are overdispersed: the counts vary more than a plain Poisson model allows, so the standard
errors are widened to match.

## What should happen

1. **The same rows and strata.** The rows used are those with every model column present. The strata used are
   those with at least one case. The number of strata dropped for having none must match too.
2. **Each coefficient** within one part in a million of statsmodels'.
3. **Each standard error**, the measure of how uncertain a coefficient is, within one part in a hundred thousand.
   It is widened by the Pearson chi-square over the residual degrees of freedom, a standard way to allow for
   overdispersion.

These criteria were fixed before the check was run.

In [ ]:
# Only what this notebook imports. See 00-kernel-check for what this kernel really has.
NEEDED = ["pandas", "statsmodels"]
try:
    import pyodide_js
    await pyodide_js.loadPackage(NEEDED)
    print("Requested:", NEEDED)
except ImportError:
    print("Not Pyodide; using the environment as installed.")

In [ ]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd()))
from checks.check_37_regression_strata import run

report = run(Path.cwd() / "fixtures" / "regression-strata-brazil-001.json")
print("agrees:", report["agrees"])
for c in report["comparisons"]:
    print(f"\n{c['note']}: rows {c['counts']['rowsUsed']}, strata {c['counts']['strataUsed']}, dropped {c['counts']['strataDropped']}")
    for e in c["estimates"]:
        print(f"  {'OK ' if e['agrees'] else 'DIFF'} {e['term']}: coefficient {e['coefficient'][0]:.10g} / {e['coefficient'][1]:.10g}"
              f", se {e['se'][0]:.8g} / {e['se'][1]:.8g}")

## The result on record

First recorded on 7 November 2026, against release 0.3.0. Rerun on 26 November 2026, against release 0.9.0:
**Agrees**.

- **Rows and strata.** Both sides used 1,613 rows and 364 strata, and dropped 16 strata.
- **Temperature.** The coefficient is 0.0375 on both sides, with a standard error of 0.0067.
- **Rainfall.** The coefficient is −0.00012 on both sides, with a standard error of 0.00021.
- **Widening.** The dispersion factor is 5.47 on both sides.

All agree far inside the criteria.

Running the cell above repeats the check on the same data.

## What this does not show

- That municipality, year and month are the right comparison for these data.
- That `tempmed` and `precip_total_semana` measure what a mosquito meets. The temperature is InfoDengue's.
  The rainfall is CHIRPS at one grid cell, summed over each week with missing days counted as 0.

In [ ]:
for item in report["notEstablished"]:
    print("-", item)
print()
print("fixture     :", report["fixture"]["id"], report["fixture"]["sha256"][:16])
print("statsmodels :", report["environment"]["statsmodels"])
print("runtime     :", report["environment"]["runtime"])